# Lab 3 — Word representations and sentiment classification

[Open this notebook in Colab](https://colab.research.google.com/github/coastalcph/nlp-course/blob/master/labs/notebooks_2026/lab_3.ipynb), then select **File → Save a copy in Drive**.

This lab compares count-based, GloVe and subword representations using the same transparent logistic-regression classifier.

## Learning objectives

By the end of this lab, you should be able to:

1. compare count-based, GloVe and subword representations;
2. train and evaluate a linear classification baseline;
3. diagnose how representation choice changes errors and learned features; and
4. select a model on validation data before evaluating it once on held-out test data.

## TA session plan

1. Run the setup and inspect the three representation types.
2. Train the same linear classifier with each representation.
3. Compare validation scores, errors and influential features.
4. Select one configuration, evaluate it once on held-out data and discuss project baselines with a TA.

# Download prerequisite packages

First we install packages that are not included in every Colab runtime: `datasets` for IMDB, `gensim` for GloVe embeddings, `bpemb` for subword embeddings, spaCy for tokenisation and scikit-learn for the fixed linear classifier.

In [ ]:
%pip install -q "datasets>=4,<5" "bpemb>=0.3,<1" "gensim>=4.3,<5" "spacy>=3.8,<4" "scikit-learn>=1.6,<2"

Here we are just using some magic commands to make sure changes to external packages are automatically loaded and plots are displayed in the notebook.

In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

# Ensuring reproducible results

Seed the random-number generators before sampling data or fitting models so everyone uses the same experimental setup.

In [ ]:
import random
import numpy as np

In [ ]:
def enforce_reproducibility(seed=42):
    random.seed(seed)
    np.random.seed(seed)

In [ ]:
enforce_reproducibility()

# Representations in NLP

- We need some way to represent our text numerically when working with ML systems
- Transform our input text into a vector or a sequence of vectors depending on the method used
- How to encode individual tokens?
- One way is to have a vocabulary of all possible tokens, assign each token a number, and encode them as a one-hot vector
![](https://raw.githubusercontent.com/copenlu/stat-nlp-book/master/img/sparse_binary.svg)
- Couple of problems
  - Inefficient: defines one feature for every word in the vocabulary
  - All words are orthogonal to each other, so what model learns about one word doesn't apply to similar words
- Solution: continuous word representations
  - Define each token in your vocab to be a $d$-dimensional vector
  - Train these vectors such that similar words have closer vectors (e.g. as measured by cosine distance)
    - Methods which do this are based on the "distributional hypothesis": similar words appear in similar contexts
    - E.g. word2vec trains word vectors to be able to predict the surrounding words (or vice-versa)
    - word2vec vectors: https://projector.tensorflow.org/
- Here we'll look at how to load/use two types of word embeddings: GloVe and byte-pair encoding (BPE) embeddings
  - GloVe is trained on word tokens using global word co-occurrence statistics ([description here](https://nlp.stanford.edu/projects/glove/#:~:text=The%20training%20objective%20of%20GloVe,'%20probability%20of%20co%2Doccurrence.&text=For%20this%20reason%2C%20the%20resulting,examined%20in%20the%20word2vec%20package.))
  - BPE works by iteratively building a vocabulary of size N via merging the most frequent character n-grams from a large text corpus (e.g. Wikipedia). The resulting tokens aren't necessarily words, but are _word pieces_. As a result, you no longer have a problem where a given token doesn't appear in your vocabulary -- you can always deconstruct a word into one or more word pieces. The embeddings for these word pieces are trained using GloVe in the `bpemb` package. Read more [here](https://github.com/bheinzerling/bpemb)

## GloVe embeddings

- For this we can use `gensim` which has a large variety of pre-trained word embeddings
- Check out their docs https://radimrehurek.com/gensim/auto_examples/index.html#documentation
- We first load 100-d embeddings trained on Wikipedia and the Gigaword corpus
- We can then examine what are the most similar words to some given words

In [ ]:
import gensim.downloader

print(list(gensim.downloader.info()['models'].keys()))

In [ ]:
# Load the vectors

glove_vectors = gensim.downloader.load('glove-wiki-gigaword-100')

In [ ]:
# Look at some of the most similar words
glove_vectors.most_similar('great')

In [ ]:
# Get vector representation of word
glove_vectors['great']

In [ ]:
glove_vectors.most_similar('bad')

## BPEmb embeddings

- Here we use `bpemb`, which has pretrained BPE tokenizers/embeddings for 275 languages
- https://github.com/bheinzerling/bpemb
- First we load the English model with 25,000 word pieces and 100-dimensions
- The package has some similar functionality built in to `gensim`, for example observing the top similar words to a given word

In [ ]:
from bpemb import BPEmb

# Load english model with 25k word-pieces
bpemb_en = BPEmb(lang='en', dim=100, vs=25000)

In [ ]:
bpemb_en.most_similar('great')

In [ ]:
bpemb_en['great']

In [ ]:
bpemb_en.most_similar('bad')

# Sentiment classification of movie reviews

We will train classifiers on the [Stanford IMDB movie-review dataset](https://huggingface.co/datasets/stanfordnlp/imdb). The notebook downloads the dataset directly, so no manual file upload is required.

In [ ]:
import spacy
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import (
    CountVectorizer,
    ENGLISH_STOP_WORDS,
    TfidfVectorizer,
)
from sklearn.metrics import accuracy_score, classification_report, f1_score

In [ ]:
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")
train_validation = imdb["train"].train_test_split(test_size=0.2, seed=42)

train_data = train_validation["train"].to_pandas()[["text", "label"]]
valid_data = train_validation["test"].to_pandas()[["text", "label"]]
test_data = imdb["test"].to_pandas()[["text", "label"]]

print(len(train_data), len(valid_data), len(test_data))

The fixed seed gives every group the same training/validation split. The official test set remains untouched until evaluation.

In [ ]:
valid_data.head()  # label 1 is positive; label 0 is negative

In [ ]:
train_data.values[0]

This defines some functions for vectorizing our text. For GloVe and BPE embeddings, this involves:

1. Tokenizing the text
2. Looking up the embedding for each token
3. Pooling the representations to obtain a single vector for the whole document

There are many options for pooling e.g. max-pooling, average-pooling, summation, etc.

In [ ]:
from tqdm import tqdm
stop_words = set(ENGLISH_STOP_WORDS)


def get_unigram_features(dataset, vectorizer):
  X = vectorizer.transform(dataset[:,0])
  y = list(dataset[:,1])
  return X,y

def get_bpemb_features(dataset, bpemb):
  # With bpemb we can tokenize and embed an entire document using .embed(x)
  X = [bpemb.embed(x).mean(0) for x in tqdm(dataset[:,0])]
  y = list(dataset[:,1])
  return X,y

def get_glove_features(dataset, glove, nlp):
  X = []
  for x in tqdm(dataset[:,0]):
    # For glove embeddings, we first tokenize the sentence using spacy, lower-case and remove stopwords, and get the available word vectors
    vecs = np.vstack([glove[t.text.lower()] if t.text.lower() in glove else np.zeros(100) for t in nlp(x) if t.text.lower() not in stop_words])
    X.append(vecs.mean(0))
  y = list(dataset[:,1])
  return np.stack(X),y

Fit the same logistic-regression classifier for every representation. Keeping the classifier fixed makes the representation comparison interpretable.

In [ ]:
def run_classifier(name, X_train, y_train, X_valid, y_valid):
    classifier = LogisticRegression(max_iter=1000, random_state=42)
    classifier.fit(X_train, y_train)
    predictions = classifier.predict(X_valid)
    scores = {
        "model": name,
        "accuracy": accuracy_score(y_valid, predictions),
        "macro_f1": f1_score(y_valid, predictions, average="macro"),
    }
    print(name)
    print(classification_report(y_valid, predictions, digits=3))
    return classifier, predictions, scores

## Mean GloVe representation

Average the available word embeddings in each review, then train the linear classifier.

In [ ]:
TRAIN_SAMPLE = 5000
VALID_SAMPLE = 1000
TEST_SAMPLE = 1000

nlp = spacy.blank("en")
X_glove_train, y_glove_train = get_glove_features(
    train_data.values[:TRAIN_SAMPLE], glove_vectors, nlp
)
X_glove_valid, y_glove_valid = get_glove_features(
    valid_data.values[:VALID_SAMPLE], glove_vectors, nlp
)
glove_model, glove_predictions, glove_scores = run_classifier(
    "Mean GloVe", X_glove_train, y_glove_train, X_glove_valid, y_glove_valid
)

## Mean BPEmb representation

Average the pretrained subword embeddings in each review and keep the classifier unchanged.

In [ ]:
X_bpemb_train, y_bpemb_train = get_bpemb_features(
    train_data.values[:TRAIN_SAMPLE], bpemb_en
)
X_bpemb_valid, y_bpemb_valid = get_bpemb_features(
    valid_data.values[:VALID_SAMPLE], bpemb_en
)
bpemb_model, bpemb_predictions, bpemb_scores = run_classifier(
    "Mean BPEmb", X_bpemb_train, y_bpemb_train, X_bpemb_valid, y_bpemb_valid
)

## Count representation

A bag of word counts discards order and semantic similarity, but preserves task-specific lexical evidence. How do you expect it to compare with averaged pretrained embeddings?

In [ ]:
count_vectorizer = CountVectorizer(min_df=2)
X_count_train = count_vectorizer.fit_transform(train_data["text"].iloc[:TRAIN_SAMPLE])
y_count_train = train_data["label"].iloc[:TRAIN_SAMPLE].tolist()
X_count_valid = count_vectorizer.transform(valid_data["text"].iloc[:VALID_SAMPLE])
y_count_valid = valid_data["label"].iloc[:VALID_SAMPLE].tolist()

count_model, count_predictions, count_scores = run_classifier(
    "Unigram counts", X_count_train, y_count_train, X_count_valid, y_count_valid
)

# Compare representations, not architectures

Because every learned system uses logistic regression, differences below come from the input representation rather than a different neural architecture.

In [ ]:
validation_results = pd.DataFrame(
    [glove_scores, bpemb_scores, count_scores]
).sort_values("macro_f1", ascending=False)
validation_results

## A deliberately simple baseline

Always predicting the majority label checks whether the learned systems add value beyond the label distribution.

In [ ]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_count_train, y_count_train)
dummy_predictions = dummy_model.predict(X_count_valid)
dummy_scores = {
    "model": "Majority label",
    "accuracy": accuracy_score(y_count_valid, dummy_predictions),
    "macro_f1": f1_score(y_count_valid, dummy_predictions, average="macro"),
}
pd.DataFrame([dummy_scores] + [glove_scores, bpemb_scores, count_scores]).sort_values(
    "macro_f1", ascending=False
)

# Inspect errors

Aggregate scores do not explain what a representation captures. Inspect some count-model errors and formulate one testable explanation.

In [ ]:
count_error_table = valid_data.iloc[:VALID_SAMPLE].copy()
count_error_table["prediction"] = count_predictions
count_error_table = count_error_table[
    count_error_table["label"] != count_error_table["prediction"]
]
count_error_table[["text", "label", "prediction"]].head(10)

## Inspect influential features

The count representation is sparse but transparent. Positive coefficients support label 1; negative coefficients support label 0.

In [ ]:
feature_names = count_vectorizer.get_feature_names_out()
weights = count_model.coef_[0]

top_negative = feature_names[np.argsort(weights)[:15]]
top_positive = feature_names[np.argsort(weights)[-15:][::-1]]
pd.DataFrame({"negative": top_negative, "positive": top_positive})

# Task: improve the representation

Try TF–IDF weighting and bigrams while keeping logistic regression fixed. This tests whether weighting and limited word order improve validation performance.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, max_features=50000, sublinear_tf=True
)
X_tfidf_train = tfidf_vectorizer.fit_transform(train_data["text"].iloc[:TRAIN_SAMPLE])
X_tfidf_valid = tfidf_vectorizer.transform(valid_data["text"].iloc[:VALID_SAMPLE])

tfidf_model, tfidf_predictions, tfidf_scores = run_classifier(
    "TF–IDF 1–2 grams",
    X_tfidf_train,
    y_count_train,
    X_tfidf_valid,
    y_count_valid,
)

## Select on validation data

Choose the learned configuration with the highest validation macro-F1. Do not inspect the test set while making this choice.

In [ ]:
learned_results = pd.DataFrame(
    [glove_scores, bpemb_scores, count_scores, tfidf_scores]
).sort_values("macro_f1", ascending=False)
learned_results

# Evaluate the selected model once

The code below applies the automatically selected representation and fitted classifier to a held-out test sample. In a project, freeze every choice before this step and report the sampling procedure.

In [ ]:
candidate_models = {
    "Mean GloVe": (
        glove_model,
        lambda rows: get_glove_features(rows, glove_vectors, nlp),
    ),
    "Mean BPEmb": (
        bpemb_model,
        lambda rows: get_bpemb_features(rows, bpemb_en),
    ),
    "Unigram counts": (
        count_model,
        lambda rows: get_unigram_features(rows, count_vectorizer),
    ),
    "TF–IDF 1–2 grams": (
        tfidf_model,
        lambda rows: get_unigram_features(rows, tfidf_vectorizer),
    ),
}

selected_name = learned_results.iloc[0]["model"]
selected_model, selected_features = candidate_models[selected_name]
X_test_final, y_test_final = selected_features(test_data.values[:TEST_SAMPLE])
test_predictions = selected_model.predict(X_test_final)

print(f"Selected on validation data: {selected_name}")
print(classification_report(y_test_final, test_predictions, digits=3))

# Connection to the project

The reusable lesson is experimental rather than architectural:

1. hold the classifier fixed when comparing representations;
2. include a deliberately simple baseline;
3. use validation data for choices;
4. inspect errors as well as aggregate metrics; and
5. evaluate held-out data only after freezing the pipeline.

# Further reading

* Pennington et al., [GloVe: Global Vectors for Word Representation](https://aclanthology.org/D14-1162/)
* Heinzerling & Strube, [BPEmb: Tokenization-free Pre-trained Subword Embeddings in 275 Languages](https://aclanthology.org/L18-1473/)
* Maas et al., [Learning Word Vectors for Sentiment Analysis](https://aclanthology.org/P11-1015/)